In [2]:
# SESSAO 04 - TAREFA 02
# Distribuicao das classes por origem dos dados

import requests
import pandas as pd

# Consultar a estrutura do repositorio
url = (
    "https://api.github.com/repos/"
    "petrobras/3W/git/trees/main?recursive=1"
)

resposta = requests.get(url, timeout=60)
resposta.raise_for_status()

arvore = resposta.json()

if arvore.get("truncated", False):
    raise ValueError(
        "Listagem incompleta do repositorio."
    )

# Criar inventario das instancias
registros = []

for item in arvore["tree"]:

    partes = item["path"].split("/")

    if len(partes) != 3:
        continue

    if partes[0] != "dataset":
        continue

    if partes[1] not in [str(i) for i in range(10)]:
        continue

    nome = partes[2]

    if not nome.endswith(".parquet"):
        continue

    if nome.startswith("WELL-"):
        origem = "Real"

    elif nome.startswith("SIMULATED_"):
        origem = "Simulada"

    elif nome.startswith("DRAWN_"):
        origem = "Especialista"

    else:
        origem = "Nao identificada"

    registros.append({
        "Categoria": int(partes[1]),
        "Origem": origem,
        "Arquivo": nome
    })

inventario_04 = pd.DataFrame(registros)

# Cruzar categorias com origens
tabela_recorte = pd.crosstab(
    inventario_04["Categoria"],
    inventario_04["Origem"]
)

tabela_recorte = tabela_recorte.reindex(
    index=range(10),
    columns=[
        "Real",
        "Simulada",
        "Especialista",
        "Nao identificada"
    ],
    fill_value=0
)

tabela_recorte["Total"] = (
    tabela_recorte.sum(axis=1)
)

tabela_recorte.loc["TOTAL"] = (
    tabela_recorte.sum(axis=0)
)

print("Distribuicao das categorias por origem:")
display(tabela_recorte)

print(
    "\nTotal de instancias:",
    len(inventario_04)
)

assert len(inventario_04) == 2228, (
    "Total diferente de 2228. Conferir a fonte!"
)


Distribuicao das categorias por origem:


Origem,Real,Simulada,Especialista,Nao identificada,Total
Categoria,,,,,
0,594,0,0,0,594
1,4,114,10,0,128
2,22,16,0,0,38
3,32,74,0,0,106
4,343,0,0,0,343
5,11,439,0,0,450
6,6,215,0,0,221
7,36,0,10,0,46
8,14,81,0,0,95



Total de instancias: 2228


In [3]:

# SESSAO 04 - TAREFA 03
# Identificacao dos rotulos e periodos transientes

import configparser

url_config = (
    "https://raw.githubusercontent.com/"
    "petrobras/3W/main/dataset/dataset.ini"
)

resposta = requests.get(
    url_config,
    timeout=30
)

resposta.raise_for_status()

config = configparser.ConfigParser(
    interpolation=None
)

config.read_string(resposta.text)

nomes = config["EVENTS"]["NAMES"].split(",")

informacoes_rotulos = []

for nome in nomes:

    nome = nome.strip()

    codigo = config.getint(
        nome,
        "LABEL"
    )

    possui_transiente = config.getboolean(
        nome,
        "TRANSIENT",
        fallback=False
    )

    codigo_transiente = (
        codigo + 100
        if possui_transiente else None
    )

    informacoes_rotulos.append({
        "Classe": codigo,
        "Evento": nome,
        "Possui_transiente": possui_transiente,
        "Codigo_transiente": codigo_transiente
    })

tabela_rotulos = pd.DataFrame(
    informacoes_rotulos
)

tabela_rotulos = tabela_rotulos.sort_values(
    "Classe"
)

display(tabela_rotulos)

print(
    "\nEventos com transiente:",
    tabela_rotulos["Possui_transiente"].sum()
)


,Classe,Evento,Possui_transiente,Codigo_transiente
0,0,NORMAL,False,NaN
1,1,ABRUPT_INCREASE_OF_BSW,True,101.0
2,2,SPURIOUS_CLOSURE_OF_DHSV,True,102.0
3,3,SEVERE_SLUGGING,False,NaN
4,4,FLOW_INSTABILITY,False,NaN
5,5,RAPID_PRODUCTIVITY_LOSS,True,105.0
6,6,QUICK_RESTRICTION_IN_PCK,True,106.0
7,7,SCALING_IN_PCK,True,107.0
8,8,HYDRATE_IN_PRODUCTION_LINE,True,108.0
9,9,HYDRATE_IN_SERVICE_LINE,True,109.0



Eventos com transiente: 7


In [4]:

# SESSAO 04 - TAREFA 04
# Distribuicao dos eventos entre pocos reais

# Selecionar apenas arquivos reais
dados_reais = inventario_04[
    inventario_04["Origem"] == "Real"
].copy()

# Extrair o identificador do poco
dados_reais["Poco"] = (
    dados_reais["Arquivo"]
    .str.extract(r"^(WELL-\d+)_")[0]
)

# Conferir se todos foram identificados
print("Arquivos reais:", len(dados_reais))

print(
    "Identificadores ausentes:",
    dados_reais["Poco"].isna().sum()
)

print(
    "Pocos distintos:",
    dados_reais["Poco"].nunique()
)

# Calcular a distribuicao por categoria
resumo_pocos = (
    dados_reais
    .groupby("Categoria")
    .agg(
        Instancias_reais=("Arquivo", "count"),
        Pocos_distintos=("Poco", "nunique")
    )
    .reindex(range(10), fill_value=0)
    .reset_index()
)

print("\nDistribuicao por categoria:")
display(resumo_pocos)

# Criar matriz de pocos x categorias
matriz_pocos = pd.crosstab(
    dados_reais["Poco"],
    dados_reais["Categoria"]
)

matriz_pocos = matriz_pocos.reindex(
    columns=range(10),
    fill_value=0
)

print("\nMatriz de pocos por categoria:")
display(matriz_pocos)

# Conferencias de consistencia
assert len(dados_reais) == 1119
assert dados_reais["Poco"].notna().all()

print("\nConferencias concluidas.")


Arquivos reais: 1119
Identificadores ausentes: 0
Pocos distintos: 40

Distribuicao por categoria:


,Categoria,Instancias_reais,Pocos_distintos
0,0,594,9
1,1,4,3
2,2,22,7
3,3,32,2
4,4,343,7
5,5,11,3
6,6,6,2
7,7,36,6
8,8,14,9
9,9,57,15



Matriz de pocos por categoria:


Categoria,0,1,2,3,4,5,6,7,8,9
Poco,,,,,,,,,,
WELL-00001,93,1,0,1,36,0,0,1,0,0
WELL-00002,209,1,1,0,112,0,3,0,0,0
WELL-00003,26,0,3,0,0,0,0,0,0,0
WELL-00004,12,0,0,0,43,0,3,0,0,0
WELL-00005,81,0,0,0,38,0,0,0,0,0
WELL-00006,113,2,0,0,0,0,0,2,0,0
WELL-00007,2,0,0,0,10,0,0,0,0,0
WELL-00008,57,0,0,0,0,0,0,0,0,0
WELL-00009,0,0,1,0,0,0,0,0,0,0



Conferencias concluidas.


In [5]:

# SESSAO 04 - TAREFA 05
# Particao preliminar dos pocos reais

from sklearn.model_selection import GroupShuffleSplit

# Lista dos pocos reais distintos
lista_pocos = pd.DataFrame({
    "Poco": sorted(dados_reais["Poco"].unique())
})

# ETAPA 1 - Separar os pocos de teste
divisao_teste = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42
)

idx_restante, idx_teste = next(
    divisao_teste.split(
        lista_pocos,
        groups=lista_pocos["Poco"]
    )
)

pocos_restantes = lista_pocos.iloc[
    idx_restante
].reset_index(drop=True)

pocos_teste = lista_pocos.iloc[
    idx_teste
]["Poco"].tolist()

# ETAPA 2 - Separar treino e validacao
divisao_validacao = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15 / 0.85,
    random_state=43
)

idx_treino, idx_validacao = next(
    divisao_validacao.split(
        pocos_restantes,
        groups=pocos_restantes["Poco"]
    )
)

pocos_treino = pocos_restantes.iloc[
    idx_treino
]["Poco"].tolist()

pocos_validacao = pocos_restantes.iloc[
    idx_validacao
]["Poco"].tolist()

# ETAPA 3 - Identificar o conjunto de cada poco
mapa_particoes = {}

for poco in pocos_treino:
    mapa_particoes[poco] = "Treino"

for poco in pocos_validacao:
    mapa_particoes[poco] = "Validacao"

for poco in pocos_teste:
    mapa_particoes[poco] = "Teste"

particoes_reais = dados_reais.copy()

particoes_reais["Conjunto"] = (
    particoes_reais["Poco"].map(mapa_particoes)
)

# ETAPA 4 - Conferencias de seguranca
assert particoes_reais["Conjunto"].notna().all()

assert set(pocos_treino).isdisjoint(pocos_validacao)
assert set(pocos_treino).isdisjoint(pocos_teste)
assert set(pocos_validacao).isdisjoint(pocos_teste)

print("Quantidade de pocos:")
print("Treinamento:", len(pocos_treino))
print("Validacao:", len(pocos_validacao))
print("Teste:", len(pocos_teste))

print("\nQuantidade de instancias:")
print(
    particoes_reais["Conjunto"].value_counts()
)

# ETAPA 5 - Verificar cobertura das categorias
cobertura = pd.crosstab(
    particoes_reais["Categoria"],
    particoes_reais["Conjunto"]
)

cobertura = cobertura.reindex(
    index=range(10),
    columns=["Treino", "Validacao", "Teste"],
    fill_value=0
)

print("\nInstancias por categoria e conjunto:")
display(cobertura)

# Verificar quantas categorias estao ausentes
print("\nCategorias ausentes no treinamento:")
print(
    cobertura.index[
        cobertura["Treino"] == 0
    ].tolist()
)

print("\nCategorias ausentes no teste:")
print(
    cobertura.index[
        cobertura["Teste"] == 0
    ].tolist()
)

print("\nPARTICAO PRELIMINAR - NAO FINALIZADA")


Quantidade de pocos:
Treinamento: 28
Validacao: 6
Teste: 6

Quantidade de instancias:
Conjunto
Treino       822
Validacao    156
Teste        141
Name: count, dtype: int64

Instancias por categoria e conjunto:


Conjunto,Treino,Validacao,Teste
Categoria,,,
0,455,57,82
1,4,0,0
2,19,2,1
3,32,0,0
4,222,83,38
5,1,6,4
6,6,0,0
7,28,0,8
8,7,2,5



Categorias ausentes no treinamento:
[]

Categorias ausentes no teste:
[1, 3, 6]

PARTICAO PRELIMINAR - NAO FINALIZADA


In [6]:

# SESSAO 04 - TAREFA 05
# Verificacao das classes menos representadas

# Selecionar as categorias com poucos pocos reais
classes_criticas = [1, 3, 5, 6]

# Filtrar somente essas categorias
dados_criticos = particoes_reais[
    particoes_reais["Categoria"].isin(
        classes_criticas
    )
].copy()

# Contar os pocos por classe e conjunto
resumo_critico = (
    dados_criticos
    .groupby(["Categoria", "Conjunto"])["Poco"]
    .nunique()
    .unstack(fill_value=0)
)

resumo_critico = resumo_critico.reindex(
    index=classes_criticas,
    columns=["Treino", "Validacao", "Teste"],
    fill_value=0
)

print("Pocos distintos por classe e conjunto:")
display(resumo_critico)

# Identificar quais pocos possuem os eventos
detalhamento = (
    dados_criticos
    .groupby(["Categoria", "Poco", "Conjunto"])
    .size()
    .reset_index(name="Instancias")
    .sort_values(["Categoria", "Poco"])
)

print("\nDistribuicao detalhada dos pocos:")
display(detalhamento)

print("\nAnalise preliminar concluida.")


Pocos distintos por classe e conjunto:


Conjunto,Treino,Validacao,Teste
Categoria,,,
1,3,0,0
3,2,0,0
5,1,1,1
6,2,0,0



Distribuicao detalhada dos pocos:


,Categoria,Poco,Conjunto,Instancias
0,1,WELL-00001,Treino,1
1,1,WELL-00002,Treino,1
2,1,WELL-00006,Treino,2
3,3,WELL-00001,Treino,1
4,3,WELL-00014,Treino,31
5,5,WELL-00015,Treino,1
6,5,WELL-00016,Teste,4
7,5,WELL-00020,Validacao,6
8,6,WELL-00002,Treino,3
9,6,WELL-00004,Treino,3



Analise preliminar concluida.


In [7]:

# SESSAO 04 - TAREFA 05
# Segunda proposta de particionamento

# Pocos selecionados para teste
pocos_teste_v2 = [
    "WELL-00004",
    "WELL-00006",
    "WELL-00010",
    "WELL-00014",
    "WELL-00016",
    "WELL-00019"
]

# Pocos selecionados para validacao
pocos_validacao_v2 = [
    "WELL-00003",
    "WELL-00005",
    "WELL-00020",
    "WELL-00022",
    "WELL-00024",
    "WELL-00033"
]

# Todos os pocos reais identificados
todos_pocos = set(dados_reais["Poco"].unique())

# Verificar os identificadores selecionados
assert set(pocos_teste_v2).issubset(todos_pocos)
assert set(pocos_validacao_v2).issubset(todos_pocos)

# Verificar que nao existem pocos repetidos
assert set(pocos_teste_v2).isdisjoint(
    pocos_validacao_v2
)

# Os demais pocos ficam no treinamento
pocos_treino_v2 = sorted(
    todos_pocos
    - set(pocos_teste_v2)
    - set(pocos_validacao_v2)
)

# Criar a tabela da segunda proposta
particoes_v2 = dados_reais.copy()

particoes_v2["Conjunto"] = "Treino"

particoes_v2.loc[
    particoes_v2["Poco"].isin(pocos_validacao_v2),
    "Conjunto"
] = "Validacao"

particoes_v2.loc[
    particoes_v2["Poco"].isin(pocos_teste_v2),
    "Conjunto"
] = "Teste"

# Verificar quantidades de pocos
print("Pocos por conjunto:")
print("Treino:", len(pocos_treino_v2))
print("Validacao:", len(pocos_validacao_v2))
print("Teste:", len(pocos_teste_v2))

# Verificar categorias por conjunto
cobertura_v2 = pd.crosstab(
    particoes_v2["Categoria"],
    particoes_v2["Conjunto"]
)

cobertura_v2 = cobertura_v2.reindex(
    index=range(10),
    columns=["Treino", "Validacao", "Teste"],
    fill_value=0
)

print("\nInstancias por categoria:")
display(cobertura_v2)

# Identificar categorias ausentes
for conjunto in ["Treino", "Validacao", "Teste"]:

    ausentes = cobertura_v2.index[
        cobertura_v2[conjunto] == 0
    ].tolist()

    print(
        f"Categorias ausentes em {conjunto}:",
        ausentes
    )

# Confirmar independencia entre conjuntos
assert set(pocos_treino_v2).isdisjoint(
    pocos_validacao_v2
)

assert set(pocos_treino_v2).isdisjoint(
    pocos_teste_v2
)

assert set(pocos_validacao_v2).isdisjoint(
    pocos_teste_v2
)

assert len(particoes_v2) == 1119

print("\nVerificacoes de separacao concluidas.")
print("PROPOSTA V2 - AINDA NAO DEFINITIVA")

Pocos por conjunto:
Treino: 28
Validacao: 6
Teste: 6

Instancias por categoria:


Conjunto,Treino,Validacao,Teste
Categoria,,,
0,361,107,126
1,2,0,2
2,18,3,1
3,1,0,31
4,158,38,147
5,1,6,4
6,3,0,3
7,7,27,2
8,10,0,4


Categorias ausentes em Treino: []
Categorias ausentes em Validacao: [1, 3, 6, 8]
Categorias ausentes em Teste: []

Verificacoes de separacao concluidas.
PROPOSTA V2 - AINDA NAO DEFINITIVA


In [8]:

# SESSAO 04 - TAREFA 05D
# Diagnostico da segunda proposta de particao

# Contar instancias e pocos por categoria
diagnostico = (
    particoes_v2
    .groupby(["Categoria", "Conjunto"])
    .agg(
        Instancias=("Arquivo", "count"),
        Pocos=("Poco", "nunique")
    )
    .reset_index()
)

# Organizar os resultados em uma tabela
tabela_diagnostico = diagnostico.pivot(
    index="Categoria",
    columns="Conjunto",
    values=["Instancias", "Pocos"]
)

tabela_diagnostico = tabela_diagnostico.fillna(0)

display(tabela_diagnostico.astype(int))

# Identificar classes com poucos exemplos
print("\nClasses com ate 5 instancias no treino:")

poucos_exemplos = cobertura_v2[
    cobertura_v2["Treino"] <= 5
]

display(poucos_exemplos)

# Identificar classes ausentes na validacao
print("\nClasses ausentes na validacao:")

classes_ausentes = cobertura_v2.index[
    cobertura_v2["Validacao"] == 0
].tolist()

print(classes_ausentes)

# Conferir se algum poco aparece em mais de um conjunto
repeticoes = (
    particoes_v2
    .groupby("Poco")["Conjunto"]
    .nunique()
)

assert (repeticoes == 1).all()

print("\nNenhum poco compartilhado entre conjuntos.")
print("Diagnostico V2 concluido.")


Instancias                  Pocos                 
Conjunto       Teste Treino Validacao Teste Treino Validacao
Categoria                                                   
0                126    361       107     3      4         2
1                  2      2         0     1      2         0
2                  1     18         3     1      5         1
3                 31      1         0     1      1         0
4                147    158        38     3      3         1
5                  4      1         6     1      1         1
6                  3      3         0     1      1         0
7                  2      7        27     1      3         2
8                  4     10         0     1      8         0
9                 11     39         7     3     10         2


Classes com ate 5 instancias no treino:


Conjunto,Treino,Validacao,Teste
Categoria,,,
1,2,0,2
3,1,0,31
5,1,6,4
6,3,0,3



Classes ausentes na validacao:
[1, 3, 6, 8]

Nenhum poco compartilhado entre conjuntos.
Diagnostico V2 concluido.


In [9]:

# SESSAO 04 - TAREFA 06A
# Exportacao das particoes provisórias V2

import zipfile
from google.colab import files

# Organizar as instancias reais
manifesto = particoes_v2[
    ["Poco", "Conjunto", "Categoria",
     "Origem", "Arquivo"]
].copy()

manifesto = manifesto.sort_values(
    ["Conjunto", "Poco", "Categoria", "Arquivo"]
).reset_index(drop=True)

# Criar a lista de pocos e seus conjuntos
lista_particoes = (
    manifesto[["Poco", "Conjunto"]]
    .drop_duplicates()
    .sort_values(["Conjunto", "Poco"])
    .reset_index(drop=True)
)

# Conferencias de seguranca
assert len(manifesto) == 1119
assert len(lista_particoes) == 40

assert (
    lista_particoes.groupby("Poco")["Conjunto"]
    .nunique() == 1
).all()

# Nomes dos arquivos
arquivo_pocos = "Pocos_V2_PROVISORIA.csv"
arquivo_instancias = "Instancias_V2_PROVISORIA.csv"

# Salvar em formato CSV
lista_particoes.to_csv(
    arquivo_pocos,
    index=False,
    encoding="utf-8-sig"
)

manifesto.to_csv(
    arquivo_instancias,
    index=False,
    encoding="utf-8-sig"
)

# Reunir os dois CSV em um arquivo ZIP
nome_zip = "Particoes_V2_PROVISORIAS.zip"

with zipfile.ZipFile(
    nome_zip, "w", zipfile.ZIP_DEFLATED
) as pacote:

    pacote.write(arquivo_pocos)
    pacote.write(arquivo_instancias)

print("Arquivos gerados com sucesso!")

print("\nTotal de pocos:", len(lista_particoes))
print("Total de instancias:", len(manifesto))

print("\nDistribuicao por conjunto:")
print(lista_particoes["Conjunto"].value_counts())

print("\nATENCAO: PARTICOES PROVISORIAS")

# Baixar o pacote para o computador
files.download(nome_zip)


Arquivos gerados com sucesso!

Total de pocos: 40
Total de instancias: 1119

Distribuicao por conjunto:
Conjunto
Treino       28
Teste         6
Validacao     6
Name: count, dtype: int64

ATENCAO: PARTICOES PROVISORIAS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>